# 19 · Check a document against a checklist

## The problem

Contracts, vendor policies and RFP responses get checked against a list of **required items**: is there a termination
notice period? a liability cap? a governing-law clause? Doing it by hand is slow and inconsistent. Asking an LLM to
"review this contract" gives you a nice summary, but prose is not a checklist: you can't sort it, count it, or prove
later what was checked.

## Why Jev

Each checklist item is really a **yes/no question about the same document**. Jev answers typed yes/no questions with
**calibrated probabilities** in about 100 ms, for a fraction of a cent, and all items go in **one call**. Because the
answer is a number, code can sort it into **MET / UNCERTAIN / MISSING**, send only the doubtful items to a person, and
store an **audit record** of exactly what was asked.

This is decision support for a reviewer, not automatic legal approval.

## What we'll build

```
document ──► Jev: one Noul per checklist item, ONE call ──► p ≥ 0.80  MET
              • termination_notice   • liability_cap              p ≤ 0.25  MISSING
              • confidentiality      • payment_terms              between   UNCERTAIN ──► reviewer queue
              • governing_law        • data_protection
                                                     └──► audit record (time, SDK version, hashes)
```

## What you'll learn
- **Atomic questions**: one narrow `Noul` per item beats one big "is this contract complete?" question.
- Turning probabilities into **three** states (not two), so the grey zone goes to a human.
- Building an **audit trail** so every result can be traced back to the exact document and questions.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #13.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the Jev API key (`TYPESAFE_API_KEY`) from the `.env` file at the repo root,
- creates the `jev` client (one object, reused for every call),
- imports the question types. In this notebook we use **`Noul`** (yes/no). `NoulCriteria`, `Score` and `Choice` are imported for the exercises,
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so the outputs are easy to read at a glance.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · The checklist, as atomic questions

Each item is **one** fact that is either in the document or not. That's what *atomic* means: no "and", no "or adequately
covers...". Narrow questions give sharp probabilities and an audit trail a lawyer can read line by line.

- **How a `Noul` works:** a yes/no question; `.noul` is the **probability that the answer is yes** (0 to 1).
- **`state` as a dict:** we'll send the contract as `state={"document": ...}`, and `` `document` `` in backticks inside each
  question refers to that key.

This cell only **defines** the questions. Nothing is sent to Jev yet.

In [2]:
CHECKLIST = {
    "termination_notice": "The document states a notice period for termination",
    "confidentiality": "The document includes a confidentiality obligation",
    "governing_law": "The document names the governing law or jurisdiction",
    "liability_cap": "The document limits or caps either party's liability",
    "payment_terms": "The document specifies when and how payment is due",
    "data_protection": "The document addresses personal-data protection (e.g. GDPR)",
}
QUESTIONS = {k: Noul(instructions=f"Is this true of `document`: {item}?") for k, item in CHECKLIST.items()}

print(len(QUESTIONS), "questions, e.g.:", QUESTIONS["governing_law"].instructions)

6 questions, e.g.: Is this true of `document`: The document names the governing law or jurisdiction?


## Step 2 · The document

A short services agreement. Before running anything, read it against the checklist yourself: notice period (clause 4),
confidentiality (3), liability cap (5) and payment terms (2) are there. There is **no governing-law clause** (clause 6 only
says disputes are resolved "amicably") and **nothing about personal data**.

In [3]:
CONTRACT = """SERVICES AGREEMENT
1. Services. Vendor will provide cloud hosting services as described in Exhibit A.
2. Fees. Customer shall pay all invoices within 30 days of receipt by bank transfer.
3. Confidentiality. Each party shall keep the other's Confidential Information secret and use it
   only to perform this Agreement.
4. Term. This Agreement runs for 12 months and either party may end it with 60 days' written notice.
5. Liability. Vendor's total liability shall not exceed the fees paid in the prior 12 months.
6. Miscellaneous. Disputes will be resolved amicably between the parties."""
print(CONTRACT)

SERVICES AGREEMENT
1. Services. Vendor will provide cloud hosting services as described in Exhibit A.
2. Fees. Customer shall pay all invoices within 30 days of receipt by bank transfer.
3. Confidentiality. Each party shall keep the other's Confidential Information secret and use it
   only to perform this Agreement.
4. Term. This Agreement runs for 12 months and either party may end it with 60 days' written notice.
5. Liability. Vendor's total liability shall not exceed the fees paid in the prior 12 months.
6. Miscellaneous. Disputes will be resolved amicably between the parties.


## Step 3 · Run the checklist and build the audit record

`check_document()` makes one `system_one` call and turns each probability into a status:

| probability `p` | status | meaning |
|---|---|---|
| `p ≥ MET_AT` (0.80) | `MET` | clearly present |
| `p ≤ MISSING_AT` (0.25) | `MISSING` | clearly absent |
| in between | `UNCERTAIN` | the text says *something* related, but not clearly enough: a person should look |

It also builds an **audit record**: when the check ran, which SDK version, and short SHA-256 fingerprints of the document
and of the checklist. If anyone later asks "was clause X checked, and against which version?", the hashes answer it
without storing the contract itself.

This cell defines the function only; the next one runs it.

In [4]:
import hashlib, datetime
from importlib.metadata import version

MET_AT, MISSING_AT = 0.80, 0.25


def check_document(doc, client):
    if "QUESTIONS" not in globals():
        raise RuntimeError("QUESTIONS is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(state={"document": doc}, questions=QUESTIONS)
    rows = []
    for k, item in CHECKLIST.items():
        p = r.nouls[k].noul
        status = "MET" if p >= MET_AT else "MISSING" if p <= MISSING_AT else "UNCERTAIN"
        rows.append({"item": k, "status": status, "p": round(p, 3)})
    audit = {
        "checked_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "sdk": f"typesafe-sdk {version('typesafe-sdk')}",
        "document_sha256": hashlib.sha256(doc.encode()).hexdigest()[:16],
        "questions_sha256": hashlib.sha256(repr(sorted(CHECKLIST.items())).encode()).hexdigest()[:16],
    }
    return rows, audit


def print_rows(rows):
    for row in rows:
        print(f"  [{row['status']:^9}] {row['item']:<20} {bar(row['p'])} {row['p']:.2f}")


print("check_document() ready")

check_document() ready


Run it on the contract.

In [5]:
rows, audit = check_document(CONTRACT, jev)
print_rows(rows)
print("\naudit:", audit)

  [   MET   ] termination_notice   ████████████████████ 0.99
  [   MET   ] confidentiality      ████████████████████ 0.99
  [ MISSING ] governing_law        ···················· 0.02
  [   MET   ] liability_cap        ███████████████████· 0.97
  [   MET   ] payment_terms        ███████████████████· 0.97
  [ MISSING ] data_protection      █··················· 0.06

audit: {'checked_at': '2026-10-18T07:19:02', 'sdk': 'typesafe-sdk 0.7.5', 'document_sha256': '2c2928bbcc8c8f2a', 'questions_sha256': '648b594e37f9a5eb'}


**What to notice**
- The four items we found by eye are **MET** with probabilities near 1.
- `governing_law` and `data_protection` are **MISSING**, near 0. "Resolved amicably" is not a governing-law clause, and Jev wasn't fooled by the fact that clause 6 talks about disputes.
- No item landed in UNCERTAIN on this contract: every clause is either clearly there or clearly not. Step 5 shows a vaguer clause that does.
- The audit record has no contract text in it, only fingerprints you can match later.

## Step 4 · The reviewer queue

A person only needs to look at what isn't clearly MET. This is where the saving is: the reviewer reads two lines, not six
clauses.

In [6]:
def reviewer_queue(rows):
    return [f"→ reviewer: {row['status'].lower()} '{CHECKLIST[row['item']]}'" for row in rows if row["status"] != "MET"]


print("\n".join(reviewer_queue(rows)) or "nothing to review")

→ reviewer: missing 'The document names the governing law or jurisdiction'
→ reviewer: missing 'The document addresses personal-data protection (e.g. GDPR)'


**What to notice**
- Only the two missing items reach the reviewer, each phrased as the original checklist line, so the reviewer doesn't need to know our internal keys.

## Step 5 · A vague clause: the UNCERTAIN band

Real contracts often gesture at a topic without really covering it. Here we add one clause,
*"Vendor will comply with all applicable laws, including privacy laws."* Does that "address personal-data protection"?
A careful lawyer would say: sort of, not really. That's exactly what the middle band is for.

In [7]:
VAGUE = CONTRACT + "\n7. Compliance. Vendor will comply with all applicable laws, including privacy laws."
rows_vague, _ = check_document(VAGUE, jev)
print_rows(rows_vague)
print()
print("\n".join(reviewer_queue(rows_vague)))

  [   MET   ] termination_notice   ████████████████████ 0.99
  [   MET   ] confidentiality      ████████████████████ 0.99
  [ MISSING ] governing_law        █··················· 0.03
  [   MET   ] liability_cap        ███████████████████· 0.97
  [   MET   ] payment_terms        ████████████████████ 0.98
  [UNCERTAIN] data_protection      ███████████········· 0.54

→ reviewer: missing 'The document names the governing law or jurisdiction'
→ reviewer: uncertain 'The document addresses personal-data protection (e.g. GDPR)'


**What to notice**
- `data_protection` moves from MISSING into **UNCERTAIN**, roughly a coin flip: Jev sees the privacy mention but no actual data-protection obligation.
- The calibrated middle is useful information, not a failure. Code sends it to a human instead of guessing.
- `governing_law` is still MISSING: "applicable laws" doesn't name any jurisdiction.

## Step 6 · Give it a face

Everything from here on is **presentation only**: a small style kit, a form wired to `check_document()` with
`ipywidgets`, and an HTML card with a coloured badge per item and the audit trail at the bottom. The checklist logic
above is untouched.

In [8]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

`checklist_html()` renders one document's checklist. We show the original contract as the example.

In [9]:
CK_COLOR = {"MET": SAGE, "UNCERTAIN": AMBER, "MISSING": RED}


def checklist_html(document):
    rows, audit = check_document(document, jev)
    items = "".join(f'<div style="padding:5px 0;">{badge(r["status"], CK_COLOR[r["status"]])} '
                    f'{CHECKLIST[r["item"]]} <span style="color:{SLATE};font-size:12px;">({r["p"]:.2f})</span></div>'
                    for r in rows)
    met = sum(r["status"] == "MET" for r in rows)
    trail = " · ".join(f"{k}: {v}" for k, v in audit.items())
    return box(f'<div style="{CARD}"><b>{met}/{len(rows)} items met</b>{items}'
               f'<div style="margin-top:8px;font-size:11px;color:{SLATE};">audit · {trail}</div></div>')


display(HTML(checklist_html(CONTRACT)))

## Step 7 · Try it yourself

Paste any contract or policy and press **Check** (needs a live kernel). Good ones to try:
- The default: the contract plus *"7. Governing law. This Agreement is governed by the laws of Ireland."* → `governing_law` flips to **MET**.
- Replace that clause with *"7. Disputes. Any dispute will be heard by the courts of Dublin."* → naming a court is a jurisdiction, so expect **MET** again.
- Add *"8. Data protection. Vendor shall process personal data only on Customer's instructions and in line with the GDPR."* → `data_protection` flips to **MET**: 6/6.
- Paste something that isn't a contract at all (a recipe, a README) → nearly everything **MISSING**.

In [10]:
form_app([("document", "Document", CONTRACT + "\n7. Governing law. This Agreement is governed by the laws of Ireland.", 10)],
         checklist_html, "Check")

Output()

## Recap and exercises

**Key ideas**
1. **Atomic questions.** One narrow `Noul` per checklist item gives sharp, auditable answers, all in one call.
2. **Three states, not two.** MET / UNCERTAIN / MISSING lets calibrated doubt go to a person instead of being forced into yes or no.
3. **Audit trail.** Timestamp, SDK version and hashes of the document and checklist make every result traceable.
4. **Decision support, not approval.** The reviewer queue is the product: it shrinks a full read to a few lines.

**Exercises**
- Add a checklist item `auto_renewal` ("The document says whether the agreement renews automatically") and check the contract.
- Narrow the band: set `MET_AT = 0.9` and `MISSING_AT = 0.1`. Does anything move into UNCERTAIN?
- Make the audit record include each item's probability and status, then save it as one JSON line per check.
- Try two near-identical contracts and confirm their `document_sha256` values differ while `questions_sha256` stays the same.